# 03 · GUE Comparison: Zeta vs Poisson vs Wigner Surmise

This notebook compares locally normalized zeta-zero spacings against two standard reference classes:

1. **Poisson / exponential spacings**  
2. **GUE-style spacings via the Wigner surmise**

It also compares a local three-zero neighborhood statistic across these sequences.

## Why this matters

A central theme in numerical work around zeta zeros is that their unfolded spacing statistics look much closer to **GUE-like behavior** than to **Poisson** behavior.

This notebook is exploratory:

- zeta zeros come from `mpmath`
- unfolding uses a local moving-average normalization
- GUE is approximated by the Wigner surmise (not full random-matrix simulation)
- no theorem-level claims are made from finite numerics

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Collect nontrivial zeros and build local-normalized spacings

In [ ]:
N = 300
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
gaps = np.diff(t)

window = 15
kernel = np.ones(window) / window
local_means = np.convolve(gaps, kernel, mode="same")

half = window // 2
for i in range(half):
    local_means[i] = gaps[:i + half + 1].mean()
    local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()

zeta_local = gaps / local_means

len(zeta_local), zeta_local.mean(), zeta_local.std()

## Reference controls

### Poisson / exponential

For a Poisson point process, nearest-neighbor spacings are exponentially distributed with mean 1.

### Wigner surmise (GUE-style proxy)

For comparison, we use the standard Wigner-surmise-type density for GUE-like level repulsion:

\[
p(s) = \frac{32}{\pi^2}s^2 e^{-4s^2/\pi},
\qquad s \ge 0.
\]

This is not a full GUE simulation, but it is a standard reference shape for spacing comparisons.

> **Note:** The GUE comparison here uses the Wigner surmise as a proxy, not full random-matrix simulation.

In [ ]:
M = len(zeta_local)

poisson_spacings = rng.exponential(scale=1.0, size=M)

def wigner_gue_pdf(s):
    return (32 / np.pi**2) * s**2 * np.exp(-4 * s**2 / np.pi)

def sample_wigner_gue(size, rng, s_max=3.5):
    grid = np.linspace(0, s_max, 2000)
    pdf_max = wigner_gue_pdf(grid).max()
    out = []
    while len(out) < size:
        s = rng.uniform(0, s_max, size=size)
        u = rng.uniform(0, pdf_max, size=size)
        accepted = s[u <= wigner_gue_pdf(s)]
        out.extend(accepted.tolist())
    return np.array(out[:size])

gue_spacings = sample_wigner_gue(M, rng)

## Histogram: zeta vs Poisson vs GUE-style reference

In [ ]:
plt.figure(figsize=(8.8, 5.2))
bins = np.linspace(0, 3.5, 28)
plt.hist(zeta_local, bins=bins, density=True, alpha=0.55, label="zeta local norm")
plt.hist(poisson_spacings, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_spacings, bins=bins, density=True, alpha=0.45, label="GUE (Wigner)")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Spacing comparison")
plt.legend()
plt.show()

## Overlay reference curves

In [ ]:
s = np.linspace(0, 3.5, 400)
poisson_pdf = np.exp(-s)
gue_pdf = (32 / np.pi**2) * s**2 * np.exp(-4 * s**2 / np.pi)

plt.figure(figsize=(8.8, 5.2))
plt.hist(zeta_local, bins=np.linspace(0, 3.5, 28), density=True, alpha=0.45)
plt.plot(s, poisson_pdf, linewidth=2, label="Poisson pdf")
plt.plot(s, gue_pdf, linewidth=2, label="GUE pdf")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Zeta vs reference curves")
plt.legend()
plt.show()

## Local three-zero neighborhood statistic

In [ ]:
def local_balance(seq):
    left = seq[:-1]
    right = seq[1:]
    return np.minimum(left, right) / np.maximum(left, right)

zeta_balance = local_balance(zeta_local)
poisson_balance = local_balance(poisson_spacings)
gue_balance = local_balance(gue_spacings)

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 1, 24)
plt.hist(zeta_balance, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_balance, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_balance, bins=bins, density=True, alpha=0.45, label="GUE")
plt.xlabel("balance")
plt.ylabel("density")
plt.title("Triplet balance comparison")
plt.legend()
plt.show()